---
# Example Notebook 01 - Prep WFSS Data for extraction

This notebook bulk prepares NIRCam WFSS _rate files downloaded from MAST for extraction with grizli. It can handle any combination of filters/grisms/modules and fields. This notebook is one of two, with the second notebook extracting spectra based on a target catalog.

To optimally run grizli you need the following ancillary files:
- A segmentation image, ideally from a LW stack
- A detection catalogue from the same LW stack
- Mosaics for every filter you are perfoming extraction on

**This example process the available WFSS data in Abell 2744. These programs are [ALT](https://arxiv.org/abs/2410.01874v1) (PI. Naidu), [MAGNIF](https://arxiv.org/abs/2503.03829) (PI. Sun) and GO 3538 (PI. Iani).  
The direct images used in this example are produced by Notebook 00a and are retrieved from the DJA. The source catalog is produced by the [UNCOVER](https://scixplorer.org/abs/2024ApJ...974...92B/abstract?fromADS=true&ads_compat=1) team.**

---

### Note: If you are reducing a lot of data you should convert this notebook into a .py file and run natively.

In [8]:
# Import some things
import os
import warnings

import re
import shutil
import multiprocessing as mp
import copy


N_PROC = 32 # Multiprocessing cores, make sure you change this

# Set the base directory for all files etc.
WORK_ROOT = '/data/users/cooper/WFSS_GRIZLI_NEW'
print(WORK_ROOT)


# Environment variables
# Note: On my remote server I have to define these in terminal before running the notebook, so these lines are redundant
# os.environ["CRDS_PATH"] = "YOUR_PATH_HERE"
# os.environ["CRDS_SERVER_URL"] = "https://jwst-crds.stsci.edu"

# Choose which version of grizlis config files to use
os.environ['NIRCAM_CONF_VERSION'] = 'V9'

# Build subdirectiories if they don't exist (These directories must exist for grizli to function properly)
for sub in ["Raw", "Prep", "Extractions", "Persistence"]:
    os.makedirs(os.path.join(WORK_ROOT, sub), exist_ok=True)

# Import necessary modules
import numpy as np
import matplotlib.pyplot as plt
import astropy
from astropy.io import fits
from astropy.table import Table
import glob

import grizli
from grizli import utils, prep, jwst_utils, multifit, fitting
from grizli.pipeline import auto_script

import jwst


# Remove some nuisance warnings
utils.set_warnings()           
warnings.filterwarnings("ignore", category=RuntimeWarning)

os.chdir(WORK_ROOT)

/data/users/cooper/WFSS_GRIZLI_NEW


In [9]:
# I have some issues with flat files not being found, so I redefine the jwst_utils download to work over https
# You may not need this and can comment it out, but keep it in if you get a "Credentials Error"
SKYFLAT_DIR = os.path.join(grizli.GRIZLI_PATH, "CONF", "NircamSkyFlat")
os.makedirs(SKYFLAT_DIR, exist_ok=True)
print(f"Sky flats will be stored in {SKYFLAT_DIR}")

def _download_skyflat_https(skyfile, bucket_prefix="grizli-v2/NircamSkyflats/", **kwargs):
    url = "https://s3.amazonaws.com/" + bucket_prefix + os.path.basename(skyfile)
    local_file, status = utils.general_fetch_file(url, path=os.path.dirname(skyfile), cache=True)
    return local_file if status > 0 else None

jwst_utils.download_skyflat_file = _download_skyflat_https

Sky flats will be stored in /Users/users/cooper/.local/lib/python3.9/site-packages/grizli/data/CONF/NircamSkyFlat


In [10]:
# Produce an overview of all available files
RAW_SOURCE_DIR = os.path.abspath("./Raw")
rate_files = sorted(glob.glob(os.path.join(RAW_SOURCE_DIR, "jw03538*_rate.fits")))
print(f"Found {len(rate_files)} rate files\n")

keys = ["PROGRAM", "TARGPROP", "OBSERVTN", "DETECTOR", "FILTER", "PUPIL"]
rows = [[fits.getval(f, k, ext=0) for k in keys] for f in rate_files]
info = Table(rows=rows, names=keys)
info["file"] = rate_files

summary = info.group_by(["PROGRAM", "TARGPROP", "FILTER", "PUPIL"])
for key, group in zip(summary.groups.keys, summary.groups):
    print(f"{key['PROGRAM']:>6}  {key['TARGPROP']:<20}  {key['FILTER']:>6}  {key['PUPIL']:>7}  {len(group):4d} files")

Found 314 rate files

 03538  A2744                  F300M    CLEAR    18 files
 03538  A2744                  F300M   GRISMC     8 files
 03538  A2744                  F300M   GRISMR     4 files
 03538  A2744                  F335M    CLEAR    24 files
 03538  A2744                  F335M   GRISMC     8 files
 03538  A2744                  F335M   GRISMR     8 files
 03538  A2744                  F410M    CLEAR    22 files
 03538  A2744                  F410M   GRISMC     8 files
 03538  A2744                  F410M   GRISMR     8 files
 03538  A2744                  F460M    CLEAR    24 files
 03538  A2744                  F460M   GRISMC     8 files
 03538  A2744                  F460M   GRISMR     8 files
 03538  A2744-Tile-2           F300M    CLEAR     6 files
 03538  M0416                  F300M    CLEAR    24 files
 03538  M0416                  F300M   GRISMC     8 files
 03538  M0416                  F300M   GRISMR     8 files
 03538  M0416                  F335M    CLEAR    2

## 01. Split rate files by target field and set-up grizli friendly directories. Perform initial flat-fielding and WCS alignment.

The following cells split all your downloaded files into separate targets. You will end up with a file-tree as follows:

```
Base
 │
 ├── Target 1
 │       ├──> Raw
 │       ├──> Prep
 │       └──> Extractions
 ├── Target 2
 │       ├──> Raw
 │       ├──> Prep
 │       └──> Extractions
 └── Etc.

```

**NOTE:** Since the "target" of an observing program is defined by what the PI sets in the APT, different observations of the same target can have different names (see the output of the cell above). In order to successfully group observations by field, you must provide user defined "target" with an alias in FIELD_MAP. The cell below will then copy all files into directories based on this mapping.

**NOTE 2:** Depending on the computing power available and the size of the data reduction, this cell may take some time.

In [11]:
# Give every field the same name
FIELD_MAP = {"A2744": "a2744", "ABELL2744": "a2744", "A2744-Tile-2": "a2744", "M0416" : "m0416", "MCS-J0416.1-2403": "m0416"}

def field_root(targprop):
    name = FIELD_MAP.get(targprop, targprop)
    return re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")

def init_grism_file(path):
    try:
        jwst_utils.initialize_jwst_image(path, oneoverf_correction=False, verbose=False)
        jwst_utils.set_jwst_to_hst_keywords(path, reset=True, verbose=False)
        return path, None
    except Exception as e:
        if os.path.exists(path):
            os.remove(path)
        return path, repr(e)

# Separate files into target folders (I am limiting this to a2744 for now)
info["FIELD"] = [field_root(t) for t in info["TARGPROP"]]
to_init = []

FIELDS_TO_RUN = ["a2744"]
info = info[np.isin(info["FIELD"], FIELDS_TO_RUN)]

print(f"Unique targets are {np.unique(info['FIELD'])}.")

for field in np.unique(info["FIELD"]):
    print(f"Copying files for {field}")
    for sub in ["RAW", "Prep", "Extractions"]:
        os.makedirs(os.path.join(WORK_ROOT, field, sub), exist_ok=True)

    n_copied = 0
    for row in info[info["FIELD"] == field]: # <----
        dest = os.path.join(WORK_ROOT, field, "RAW", os.path.basename(row["file"]))
        if os.path.exists(dest):
            continue
        shutil.copy2(row["file"], dest)
        n_copied += 1
        if "GRISM" in row["PUPIL"]:
            to_init.append(dest)
    print(f"{field}: copied {n_copied} files")

print(f"\n{len(to_init)} grism exposures to initialise with {N_PROC} processes")


# Initialise the files ready to work with grizli
results = []
if to_init:
    results.append(init_grism_file(to_init[0]))

    with mp.get_context("fork").Pool(N_PROC) as pool:
        for i, res in enumerate(pool.imap_unordered(init_grism_file, to_init[1:]), start=2):
            results.append(res)
            print(f"\r  {i}/{len(to_init)} done", end="")

failed = [(p, e) for p, e in results if e]
print(f"\n\nFinished: {len(results) - len(failed)} initialised, {len(failed)} failed")
for path, err in failed:
    print(f"  FAILED {os.path.basename(path)}: {err}")

Unique targets are FIELD
-----
a2744.
Copying files for a2744
a2744: copied 0 files

0 grism exposures to initialise with 32 processes


Finished: 0 initialised, 0 failed


## 02. Begin preprocessing with GRIZLI
**NOTE:** You will need a master catalog for each field (defined in MASTER_CATS) that contains the ra and dec of objects in your field. This is used primarily for alignment, so the WCS of your catalogs should be secure. Here, I am using the catalogs provided by UNCOVER and CANUCS for A2744 and M0416 respectively.

New catalog files (with RA and DEC) only are saved in each fields directory to be passed to the pre-processing steps.

In [12]:
MASTER_CATS = {
    "a2744": f"{WORK_ROOT}/Catalogues/UNCOVER_DR3_LW_SUPER_catalog.fits",
    #"m0416": f"{WORK_ROOT}/Catalogues/hlsp_canucs_jwst-hst_multi_macs0416-clu_multi_v1_photometry-cat.fits",
}

for field, cat_file in MASTER_CATS.items():
    cat = Table.read(cat_file)
    out = os.path.join(WORK_ROOT, field, f"{field}_master.radec")
    try:
        np.savetxt(out, np.column_stack([cat['RA'], cat['DEC']]),
               fmt="%.8f", header="ra dec")
    except:
        np.savetxt(out, np.column_stack([cat['ra'], cat['dec']]),
               fmt="%.8f", header="ra dec")
    print(f"{field}: {len(cat)} sources -> {out}")

a2744: 74020 sources -> /data/users/cooper/WFSS_GRIZLI_NEW/a2744/a2744_master.radec


## Preprocessing
If you have a large amount of data, this cell can take some time. Multiprocessing can speed this process up, but requires some tweaking of certain grizli routines.

In [13]:
import copy
import contextlib
import traceback
import multiprocessing as mp
from collections import defaultdict
import drizzlepac.util

N_PROC = 32 # Set the number of CPUs here

VISIT_PREP_ARGS = copy.deepcopy(auto_script.args["visit_prep_args"])
VISIT_PREP_ARGS["tweak_max_dist"] = 1
VISIT_PREP_ARGS["use_self_catalog"] = False

"""
# Grizli creates a temporary folder called ./Tweaklock - multiprocessing causes problems here, so we lock the multiprocessing
# and allow only one process at a time to access the folder
"""
TWEAK_LOCK = mp.get_context("fork").Lock()
if not hasattr(prep, "_iterate_tweak_align_orig"):
    prep._iterate_tweak_align_orig = prep.iterate_tweak_align

def _locked_tweak_align(*args, **kwargs):
    with TWEAK_LOCK:
        return prep._iterate_tweak_align_orig(*args, **kwargs)

prep.iterate_tweak_align = _locked_tweak_align

# Parse visits to group observations for parallel processing
jobs = []
for field in np.unique(info["FIELD"]):
    prep_dir = os.path.join(WORK_ROOT, field, "Prep")
    os.makedirs(os.path.join(WORK_ROOT, field, "logs"), exist_ok=True)
    os.chdir(prep_dir)
    utils.LOGFILE = os.path.join(WORK_ROOT, field, f"{field}.grizli.log.txt")

    if not os.path.exists(f"{field}_visits.yaml"):
        raw_files = sorted(glob.glob(os.path.join("..", "RAW", "*_rate.fits")))
        lw_files = [f for f in raw_files if "LONG" in fits.getval(f, "DETECTOR", ext=0)]
        auto_script.parse_visits(files=lw_files, field_root=field, RAW_PATH="../RAW",
                                 use_visit=True, combine_same_pa=False, combine_minexp=2,
                                 max_dt=1.0, visit_split_shift=1.5)

    _, groups, _ = auto_script.load_visit_info(field, verbose=False)
    by_direct = defaultdict(list)
    for g in groups:
        if not glob.glob(g["grism"]["product"] + "_dr?_sci.fits"):
            by_direct[g["direct"]["product"]].append((g["direct"], g["grism"]))
    jobs += [(field, chain) for chain in by_direct.values()]
    print(f"{field}: {len(groups)} visit pairs, {sum(len(c) for c in by_direct.values())} still to do")

os.chdir(WORK_ROOT)

# Process visits in parallel
def run_job(job):
    field, chain = job
    name = chain[0][0]["product"]
    log_file = os.path.join(WORK_ROOT, field, "logs", f"{name}.log.txt")
    drizzlepac.util.get_pool_size = lambda *a, **k: 1    # AstroDrizzle serial inside workers
    try:
        os.chdir(os.path.join(WORK_ROOT, field, "Prep"))
        utils.LOGFILE = log_file
        master = os.path.join(WORK_ROOT, field, f"{field}_master.radec")
        with open(log_file, "a") as fh, contextlib.redirect_stdout(fh), contextlib.redirect_stderr(fh):
            for direct, grism in chain:
                prep.process_direct_grism_visit(direct=direct, grism=grism, radec=master,
                                                skip_direct=False,
                                                **copy.deepcopy(VISIT_PREP_ARGS))
                for f in direct["files"] + grism["files"]:
                    utils.fix_flt_nan(f, verbose=False)
        return name, None
    except Exception:
        return name, traceback.format_exc()

print(f"\nRunning {len(jobs)} jobs with {N_PROC} processes")
failed = []
with mp.get_context("fork").Pool(N_PROC) as pool:
    for i, (name, err) in enumerate(pool.imap_unordered(run_job, jobs), start=1):
        if err:
            failed.append((name, err))
        print(f"\r  {i}/{len(jobs)} jobs done, {len(failed)} failed", end="")

print()
for name, err in failed:
    print(f"FAILED {name}: {err.strip().splitlines()[-1]}")
os.chdir(WORK_ROOT)

a2744: 30 visit pairs, 0 still to do

Running 0 jobs with 32 processes



## Do the contamination modelling

I am modifying a number of grizli routines here to avoid modelling objects that fall outside the grizli footprint. I am also supressing the warning grizli gives when this happens.

In [ ]:
from grizli import model
import sys
from grizli import utils as grizli_utils

if not hasattr(model.GrismFLT, "_compute_model_orders_orig"):
    model.GrismFLT._compute_model_orders_orig = model.GrismFLT.compute_model_orders

def _safe_compute_model_orders(self, *args, **kwargs):
    try:
        return self._compute_model_orders_orig(*args, **kwargs)
    except KeyError:

model.GrismFLT.compute_model_orders = _safe_compute_model_orders


if not hasattr(grizli_utils, "_log_exception_orig"):
    grizli_utils._log_exception_orig = grizli_utils.log_exception
SUPPRESSED = {"trace_box": 0}

def _filtered_log_exception(LOGFILE, traceback, *args, **kwargs):
    exc_type = sys.exc_info()[0]
    if exc_type is IndexError and "process_config" in traceback.format_exc():
        SUPPRESSED["trace_box"] += 1
        return
    return grizli_utils._log_exception_orig(LOGFILE, traceback, *args, **kwargs)

grizli_utils.log_exception = _filtered_log_exception

# I was getting errors with traces falling outside grizlis default box size for refinement
# So I have modified this command to make a larger box possible for each contamination source
MIN_SIZE = 72       
REFINE_SIZE = 72   
PAD = (400, 400)

if not hasattr(multifit.GroupFLT, "_refine_orig"):
    multifit.GroupFLT._refine_orig = multifit.GroupFLT.refine

def _refine_large_box(self, *args, **kwargs):
    kwargs["size"] = max(kwargs.get("size", 30), REFINE_SIZE)
    return self._refine_orig(*args, **kwargs)

multifit.GroupFLT.refine = _refine_large_box

In [14]:
PRELIM_MAG_LIMIT = 25          # sources modelled in the first pass
REFINE_MAG_LIMITS = [18, 24]   # sources whose spectra are refined against the data
REFINE_NITER = 1
REFINE_POLY_ORDER = 3
CPU_COUNT = 36

for field in np.unique(info["FIELD"]):
    print(f"\n{'=' * 70}\n  {field}\n{'=' * 70}")
    prep_dir = os.path.join(WORK_ROOT, field, "Prep")
    os.chdir(prep_dir)
    utils.LOGFILE = os.path.join(WORK_ROOT, field, f"{field}.grizli.log.txt")

    if glob.glob("*GrismFLT.fits"):
        print("GrismFLT files exist, skipping")
        continue

    grp = auto_script.grism_prep(
        field_root=field,
        PREP_PATH="../Prep",
        EXTRACT_PATH="../Extractions",
        prelim_mag_limit=PRELIM_MAG_LIMIT,
        refine_mag_limits=REFINE_MAG_LIMITS,
        refine_niter=REFINE_NITER,
        refine_poly_order=REFINE_POLY_ORDER,
        cpu_count=CPU_COUNT,
        pad=PAD,
        model_kwargs={"compute_size": True, "min_size": MIN_SIZE},
        use_jwst_crds=False,
    )
    del grp

os.chdir(WORK_ROOT)

grizli patched: failed spectral orders are skipped
Trace-box IndexError messages will be suppressed

  a2744
# (2026-11-25 08:59:09.842)

auto_script.grism_prep(**{'field_root': 'a2744', 'PREP_PATH': '../Prep', 'EXTRACT_PATH': '../Extractions', 'ds9': None, 'refine_niter': 1, 'gris_ref_filters': {'G141': ['F140W', 'F160W', 'F125W', 'F105W', 'F110W', 'F098M', 'F127M', 'F139M', 'F153M', 'F132N', 'F130N', 'F128N', 'F126N', 'F164N', 'F167N'], 'G102': ['F105W', 'F098M', 'F110W', 'F125W', 'F140W', 'F160W', 'F127M', 'F139M', 'F153M', 'F132N', 'F130N', 'F128N', 'F126N', 'F164N', 'F167N'], 'G800L': ['F814W', 'F850LP', 'F606W', 'F435W', 'F775W', 'F555W', 'opt'], 'GR150C': ['F115W', 'F150W', 'F200W', 'F090W', 'CLEAR-F090W', 'CLEAR-F115W', 'CLEAR-F150W', 'CLEAR-F200W', 'F090WN-CLEAR', 'F115WN-CLEAR', 'F150WN-CLEAR', 'F200WN-CLEAR'], 'GR150R': ['F115W', 'F150W', 'F200W', 'F090W', 'CLEAR-F090W', 'CLEAR-F115W', 'CLEAR-F150W', 'CLEAR-F200WF090WN-CLEAR', 'F115WN-CLEAR', 'F150WN-CLEAR', 'F200WN-CLEAR'],

## Preprocessing done! You can move on to example notebook 02 for spectral extraction.